# Part 2: CIFAR-10 classification with the OpenAI Responses API

Can a zero-shot fixed-choice image classifier improve on the historical **79.08%** supervised CNN result? This notebook measures the answer; it does not assume superiority or promise the aspirational **99%**. Historical cat F1 was **0.626**, bird F1 **0.688**. These reference constants are self-contained and require no Part 1 files.

**Safe to Run all:** every paid-stage flag is off. No API client or secret is accessed unless you explicitly enable a paid stage. The default lifetime cap is **20 actual request attempts**, including retries. No model accuracy has been measured in this delivered notebook. CIFAR download and package installation are free network operations; API inference is not.

The only inference call is official SDK `client.responses.create()`, with exactly `gpt-6.1-sol`, one original image, the same question and neutral ten-class descriptions, and strict label-only JSON. No confidence, probabilities or rationale are requested.

Official OpenAI documentation checked **2026-11-07**: [requested model](https://developers.openai.com/api/docs/models/gpt-6.1-sol), [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs), [Python create reference](https://developers.openai.com/api/reference/python/resources/responses/methods/create), [image detail support](https://developers.openai.com/api/docs/guides/images-vision). The model page confirms image input, Responses and Structured Outputs; the vision table supports `original` detail for this model. Installed official SDK **3.46.0** types and mocked Responses transport were verified. Account access and successful live inference were not verified.

## 1. Install and import

This pins the tested official SDK. It preserves Colab's compatible torch/torchvision pair instead of upgrading them. Standard Colab includes both; if torchvision import fails, select a fresh standard Python runtime. Other packages provide data handling, metrics and plots only.

In [ ]:
%pip -q install "openai==3.46.0" "numpy>=1.26,<3" "pandas>=2.2,<4" "matplotlib>=3.8,<4" "seaborn==0.13.2" "Pillow>=10,<13" "tqdm>=4.66,<5"

In [ ]:
import base64
import hashlib
import io
import json
import math
import os
import random
import time
import uuid
import warnings
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import fcntl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display, Markdown
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm
import openai
from openai import OpenAI

sns.set_theme(style="whitegrid", context="notebook")
print("Official OpenAI SDK:", openai.__version__)
assert hasattr(OpenAI, "responses"), "Run the install cell and restart the runtime."

## 2. Configure execution and persistence

Start with `RUN_SMOKE10=True` only. After reviewing it, enable `RUN_PILOT100`; cached overlap is reused. Increase `MAX_API_ATTEMPTS` explicitly if needed. The cap belongs to this **output directory across reruns and sessions**, counts every reserved request even after a crash, and is not a monetary cap. Never delete attempt records to reset it. Before enabling test evaluation, turn development flags off, set `FREEZE_PROTOCOL=True`, and execute the freeze cell. Then explicitly enable evaluation. The full 10,000-image stage is separately off by default.

Set `USE_GOOGLE_DRIVE=True` **before the first run** for persistence beyond Colab VM resets. Drive mounting is optional and can request Google authorization; it never reads the OpenAI key. Use one runtime per results directory. File locking guards overlapping execution in a single runtime, not distributed synchronization between VMs. Local `/content` files disappear when Colab resets. Keep the complete results folder, especially `attempts`, `images`, `manifests`, and `protocol_freeze.json`.

Live execution reads only `google.colab.userdata.get("OPENAI_API_KEY")`. Add that name in Colab's Secrets panel and grant notebook access. Do not paste a key into a cell. All flags off also works outside Colab without Secrets. No environment-key fallback is used. Editable standard rates are **$2.00 input, $0.10 cached input, $2.50 default 30-minute cache writes and $10.00 output per million tokens**, sourced from the model page on 2026-11-07. Estimates are not bills or monetary caps. Missing usage stays unknown; no 1000-image projection is given before usable observations.

Cache-write counts are required for a priced estimate. Following [official prompt caching documentation](https://developers.openai.com/api/docs/guides/prompt-caching), subtract disjoint reads and writes from ordinary input and charge each once. Missing cache-write usage remains unknown. No cache options are sent, so writes use the documented 30-minute default; returned TTL breakdowns price 12-hour writes at twice the 30-minute rate.


In [ ]:
# All switches are intentionally off. Run all is free of API calls and secret access.
RUN_SMOKE10 = False
RUN_PILOT100 = False
FREEZE_PROTOCOL = False
RUN_EVALUATION1000 = False
RUN_FULL10000 = False
USE_GOOGLE_DRIVE = False
OUTPUT_DIR = Path("/content/cifar10_responses_sol_results") if Path("/content").exists() else Path.cwd() / "cifar10_responses_sol_results"
DATA_ROOT = Path("/content/cifar10_data") if Path("/content").exists() else OUTPUT_DIR / "dataset"
MAX_API_ATTEMPTS = 20  # Lifetime cap for this output directory, including retries.
MAX_ATTEMPTS_PER_IMAGE = 3
REQUEST_TIMEOUT_SECONDS = 60.0
MAX_BACKOFF_SECONDS = 30.0
RETRY_UNCERTAIN_REQUESTS = False  # May duplicate billing after timeout/5xx/crash.
RETRY_RATE_LIMITED = False  # Explicit resume of cached 429 exhaustion; retains prior attempts.
SAMPLING_SEED = 20261106
VALIDATION_PER_CLASS = 500
STAGE_PER_CLASS = {"smoke10": 1, "pilot100": 10, "evaluation1000": 100, "full10000": 1000}
CALIBRATION_BINS = 10
PROBABILITY_SUM_TOLERANCE = 1e-6
# Editable standard USD per 1M tokens, checked 2026-11-07 against the model page.
# Estimates, not bills or hard monetary caps; regional/other adjustments may apply.
PRICING_SOURCE = "https://developers.openai.com/api/docs/models/gpt-6.1-sol"
PRICING_CHECKED_DATE = "2026-11-07"
TOKEN_RATES_USD_PER_MILLION = {"uncached_input": 2.00, "cached_input": 0.10, "cache_write": 2.50, "output": 10.00}
HISTORICAL_CNN_ACCURACY = 0.7908

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        raise RuntimeError("Google Drive mounting requires Colab.") from None
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/cifar10_responses_sol_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for folder in ["attempts", "images", "manifests", "exports"]:
    (OUTPUT_DIR / folder).mkdir(exist_ok=True)
assert isinstance(MAX_API_ATTEMPTS, int) and MAX_API_ATTEMPTS >= 0
assert MAX_ATTEMPTS_PER_IMAGE >= 1 and MAX_BACKOFF_SECONDS >= 0
print("Results:", OUTPUT_DIR)
print("Paid stages enabled:", {"smoke10": RUN_SMOKE10, "pilot100": RUN_PILOT100,
      "evaluation1000": RUN_EVALUATION1000, "full10000": RUN_FULL10000})
print("Persistent request-attempt cap:", MAX_API_ATTEMPTS)

## 3. Fixed choices and lossless inputs

A strict JSON Schema returns exactly one object with a required `label` enum of the ten mutually exclusive CIFAR labels and `additionalProperties=false`. The unchanged question and short neutral category descriptions accompany the image as text, without examples or ground truth. This answer supplies no confidence or class distribution.

The request contains only the image and exact required question/options. The question necessarily names CIFAR-10 as requested; no per-image dataset metadata, sample ID, label, filename, split or stage enters the request. Images are original RGB **32x32 PNGs** with decoded pixel equality checked, and `detail="original"` as documented. There is no resizing or added information. The provider can still preprocess internally; this setting does not give us control of its internal image pipeline.

The cache fingerprint combines PNG content and exact endpoint namespace/model/question/descriptions/detail/schema/reasoning/output limit. The broader frozen procedure also records sampling, stage sizes, retry policies and metric settings. Use a new results directory for a different frozen study, rather than mixing configurations. Raising the lifetime attempt cap or explicitly enabling exhausted-429/uncertain resume does not change the fixed question/sampling procedure; these operational controls are recorded on each new attempt.

The requested `reasoning.effort="medium"` uses a configurable `max_output_tokens=4096` total allowance including reasoning and the final label JSON. This is a cap, not expected usage; actual recorded usage determines cost. Incomplete responses remain failed classifications and are cached without automatic retries or cap increases. Both settings are fingerprinted and frozen. No upscaling is used: larger dimensions can increase token costs and equal latency cannot be guaranteed. Sol cost and latency relative to mini are unmeasured.

In [ ]:
CLASSES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
DESCRIPTIONS = ["A powered aircraft with wings.", "A passenger car.", "A feathered animal with a beak.",
                "A domestic feline.", "A hoofed deer.", "A domestic canine.", "A frog amphibian.",
                "A horse equine.", "A waterborne vessel.", "A motor vehicle for carrying cargo."]
QUESTION = "Which of the ten CIFAR-10 categories best describes the primary object visible in this image? Choose the single most appropriate category."
MODEL = "gpt-6.1-sol"
REASONING_EFFORT = "medium"  # Requested inference effort, explicit for the fingerprint.
MAX_OUTPUT_TOKENS = 4096  # Configurable total cap including reasoning, not expected usage.
assert type(MAX_OUTPUT_TOKENS) is int and 1 <= MAX_OUTPUT_TOKENS <= 128000
IMAGE_DETAIL = "original"
CHOICES = [{"value": label, "description": description} for label, description in zip(CLASSES, DESCRIPTIONS)]
assert len(CHOICES) == len(set(CLASSES)) == 10

def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False)

def digest(value):
    return hashlib.sha256(canonical(value).encode()).hexdigest()

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)

def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)

def request_semantics():
    instructions = QUESTION + "\n\n" + "\n".join(
        item["value"] + ": " + item["description"] for item in CHOICES)
    return {"model": MODEL, "detail": IMAGE_DETAIL, "instructions": instructions,
            "reasoning": {"effort": REASONING_EFFORT}, "max_output_tokens": MAX_OUTPUT_TOKENS,
            "text": {"format": {"type": "json_schema", "name": "classification", "strict": True,
                                "schema": {"type": "object", "properties": {
                                    "label": {"type": "string", "enum": CLASSES}},
                                    "required": ["label"], "additionalProperties": False}}}}

def protocol():
    return {"version": "responses-sol-cifar10-v1", "request": request_semantics(), "seed": SAMPLING_SEED,
            "validation_per_class": VALIDATION_PER_CLASS, "stage_per_class": STAGE_PER_CLASS,
            "procedure": {"sampling": "classwise default_rng SeedSequence(seed, split_code), nested shuffled class lists",
                          "splits": "development from original train only; evaluation from original test only",
                          "image": "lossless RGB 32x32 PNG; decoded pixel equality asserted; no resizing",
                          "selection": "strict label-only JSON; confidence and probabilities unavailable", "sdk": "openai==3.46.0",
                          "retry": "429 exponential+jitter; exhausted 429 resumes only with explicit runtime opt-in; uncertain opt-in; permanent errors cached",
                          "max_attempts_per_image": MAX_ATTEMPTS_PER_IMAGE,
                          "timeout_seconds": REQUEST_TIMEOUT_SECONDS, "max_backoff_seconds": MAX_BACKOFF_SECONDS,
                          "retry_uncertain_policy": "off by default; explicit operational opt-in may retry intent/timeout/transport/408/5xx with possible duplicate billing; per-attempt consent recorded; all attempts count",
                          "calibration_bins": CALIBRATION_BINS, "probability_tolerance": PROBABILITY_SUM_TOLERANCE}}

def png_bytes(image):
    pixels = np.asarray(image)
    assert pixels.shape == (32, 32, 3) and pixels.dtype == np.uint8
    buffer = io.BytesIO()
    Image.fromarray(pixels).save(buffer, format="PNG")
    data = buffer.getvalue()
    decoded = np.asarray(Image.open(io.BytesIO(data)))
    assert np.array_equal(decoded, pixels), "PNG round-trip changed original pixels."
    return data

def request_body(png):
    semantics = request_semantics()
    return {"model": semantics["model"], "input": [{"role": "user", "content": [{
        "type": "input_image", "image_url": "data:image/png;base64," + base64.b64encode(png).decode("ascii"),
        "detail": semantics["detail"]}, {"type": "input_text", "text": semantics["instructions"]}]}],
        "text": semantics["text"], "reasoning": semantics["reasoning"],
        "max_output_tokens": semantics["max_output_tokens"]}

def cache_key(png):
    return digest({"namespace": "responses-sol-cifar10-v1", "png_sha256": hashlib.sha256(png).hexdigest(), "semantics": request_semantics()})

def freeze_path():
    return OUTPUT_DIR / "protocol_freeze.json"

def check_freeze():
    if not freeze_path().exists():
        raise RuntimeError("Freeze the development protocol in the freeze cell before enabling test evaluation.")
    if read_json(freeze_path())["protocol"] != protocol():
        raise RuntimeError("Frozen protocol differs. Restore it to resume, or use a separate output directory for a new study.")

## 4. Training-only validation pool and examples

The fixed seed selects **500 images per class** from original training data for a 5,000-image development pool. Smoke uses one per class and pilot ten per class, nested within that pool. No fitting or fine-tuning occurs. The remainder of training data is unused. The original test dataset is instantiated only after explicit evaluation enablement and protocol-freeze checks. The official torchvision archive contains both splits and its integrity checks may read archive files; development never instantiates, displays or classifies the test split.

Review the training examples before inference. Tiny images are enlarged only for human display, using nearest-neighbor interpolation.

In [ ]:
# This is the only dataset loaded during default execution and development.
train_data = CIFAR10(root=str(DATA_ROOT), train=True, download=True)
assert train_data.classes == CLASSES

def balanced_indices(targets, per_class, split):
    if split not in ("train_validation", "test"):
        raise ValueError("Unknown split")
    rng = np.random.default_rng(np.random.SeedSequence([SAMPLING_SEED, 0 if split == "train_validation" else 1]))
    targets = np.asarray(targets)
    groups = [rng.permutation(np.flatnonzero(targets == label)).tolist() for label in range(10)]
    limit = VALIDATION_PER_CLASS if split == "train_validation" else 1000
    if per_class > limit or any(len(group) < per_class for group in groups):
        raise ValueError("Not enough samples or stage exceeds the fixed pool.")
    # Interleave classes, preserving nested stages and early balanced progress.
    return [int(groups[label][rank]) for rank in range(per_class) for label in range(10)]

validation_pool_ids = balanced_indices(train_data.targets, VALIDATION_PER_CLASS, "train_validation")
print(f"Fixed training-only validation pool: {len(validation_pool_ids):,} images. Test dataset has not been loaded.")
fig, axes = plt.subplots(2, 5, figsize=(11, 4.8))
for axis, image_id in zip(axes.flat, validation_pool_ids[:10]):
    image, truth = train_data[image_id]
    axis.imshow(image, interpolation="nearest")
    axis.set_title(CLASSES[truth]); axis.axis("off")
fig.suptitle("Training validation examples before any inference (32x32 originals)")
fig.tight_layout(); plt.show(); plt.close(fig)

## 5. Persist deterministic sample identities

The saved manifests keep image/label pairs and sampling IDs locally. Classes are interleaved so an early budget stop remains approximately balanced. Pilot reuses smoke selections; full test reuses the 1,000-image test selections. Each stage is analyzed separately; nested stages must not be pooled as independent observations. PNGs and manifests allow later analysis and galleries without another API call.

In [ ]:
def prepare_stage(name, dataset, split):
    expected_split = "train_validation" if name in ("smoke10", "pilot100") else "test"
    assert split == expected_split
    assert bool(dataset.train) == (split == "train_validation")
    rows = []
    for image_id in balanced_indices(dataset.targets, STAGE_PER_CLASS[name], split):
        image, label = dataset[image_id]
        png = png_bytes(image)
        image_hash = hashlib.sha256(png).hexdigest()
        image_path = OUTPUT_DIR / "images" / (image_hash + ".png")
        if image_path.exists():
            assert image_path.read_bytes() == png
        else:
            image_path.write_bytes(png)
        rows.append({"sample_id": f"{split}:{image_id}", "index": image_id, "split": split,
                     "truth": CLASSES[label], "image_sha256": image_hash, "cache_key": cache_key(png)})
    manifest = {"stage": name, "split": split, "protocol": protocol(), "rows": rows}
    path = OUTPUT_DIR / "manifests" / (name + ".json")
    if path.exists() and read_json(path) != manifest:
        raise RuntimeError(f"{name}: saved sample/protocol mismatch. Restore configuration or select a new output directory.")
    atomic_json(path, manifest)
    return manifest

smoke_manifest = prepare_stage("smoke10", train_data, "train_validation")
pilot_manifest = prepare_stage("pilot100", train_data, "train_validation")
print("Development manifests saved locally. Labels and sample IDs never enter API requests.")

## 6. Resumable, budgeted Responses runner

Each network attempt first creates a durable **intent** record. On return, the structured SDK answer and all available usage are saved before parsing; rerunning parsing, metrics or plotting incurs no inference. Completed answers, refusals, permanent errors and exhausted rate-limit attempts are cached. To resume a cached 429 exhaustion, explicitly set `RETRY_RATE_LIMITED=True` and rerun the enabled stage, increasing the cap if necessary. The flag authorizes up to the per-image retry limit in that invocation, retains every prior attempt and cannot retry an answered/refused/permanent-error outcome. Turn it off afterward. It is an operational resume control, recorded per attempt and allowed after freeze; the frozen procedure explicitly includes this policy. Errors save only a category and HTTP status, never raw exception text, headers, key or client representation. SDK internal retries are disabled so the notebook counts each actual attempt.

Explicit HTTP 429 responses retry with bounded exponential backoff, jitter and bounded `Retry-After` (at most 30 seconds by default), stopping at both limits. Authentication/other permanent errors do not retry. A crash, timeout, transport problem, 408 or 5xx may leave **uncertain billing**. These are pending, never silently correct, and **never automatically resent by default**. A saved intent also counts against the cap. To deliberately retry uncertain work, set `RETRY_UNCERTAIN_REQUESTS=True`, acknowledging possible duplicate billing, and rerun the enabled stage under the persistent cap. This operational switch can be enabled after freeze: the frozen procedure declares the stable opt-in policy, while each new attempt records the actual consent switch. Turn it off after the intended resume. Model, question/options, image settings, sampling and analysis settings must still match the freeze. This is local deduplication, not documented server-side exactly-once billing. A later successful uncertain retry still preserves every prior attempt and any usage.

The parser reads persisted `output[]` message content, requiring a completed response, one completed assistant message and one `output_text` JSON object containing only an allowed label. Refusal content, incomplete/failed responses, empty/malformed JSON, extra or duplicate keys and wrong labels never count as correct. Raw structured Responses and usage remain available before parsing. Confidence and probability interfaces remain null/unavailable; no distributions are fabricated.

In [ ]:
@contextmanager
def runner_lock():
    # Serial notebook use only; do not run multiple VMs against one Drive folder.
    with (OUTPUT_DIR / "runner.lock").open("a") as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            raise RuntimeError("Another runner is using this output directory.") from None
        try:
            yield
        finally:
            fcntl.flock(lock, fcntl.LOCK_UN)

def attempt_records():
    return [read_json(path) for path in sorted((OUTPUT_DIR / "attempts").glob("*.json"))]

def attempt_count():
    return sum(1 for _ in (OUTPUT_DIR / "attempts").glob("*.json"))

def records_for(key):
    # Fingerprint in filenames avoids reparsing every other image's raw responses.
    paths = sorted((OUTPUT_DIR / "attempts").glob("*_" + key + ".json"))
    records = [read_json(path) for path in paths]
    assert all(record["cache_key"] == key for record in records)
    return records

def live_client():
    # No environment-variable fallback, client repr, raw exception output, or saved key.
    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError("Live inference requires Colab Secrets: add OPENAI_API_KEY and grant notebook access.") from None
    try:
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        raise RuntimeError("Enable OPENAI_API_KEY in Colab Secrets and grant this notebook access.") from None
    if not isinstance(key, str) or not key.strip():
        raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets before enabling a paid stage.") from None
    try:
        client = OpenAI(api_key=key, base_url="https://api.openai.com/v1", max_retries=0,
                        timeout=REQUEST_TIMEOUT_SECONDS)
    except Exception:
        raise RuntimeError("Client initialization failed. Verify the setup cell and Colab Secrets access.") from None
    finally:
        del key
    return client

def parse_answer(raw):
    result = {"status": "failed", "prediction": None, "api_confidence": None,
              "confidence_valid": False, "probability_valid": False,
              "probability_issue": "unavailable_label_only_response",
              "probabilities_raw": None, "probabilities": None, "selected_probability": None,
              "max_probability": None, "choice_argmax_disagreement": None}
    if not isinstance(raw, dict) or raw.get("status") != "completed":
        result["error_category"] = "response_not_completed"; return result
    output = raw.get("output")
    if not isinstance(output, list) or any(not isinstance(item, dict) for item in output):
        result["error_category"] = "malformed_output"; return result
    messages = [item for item in output if item.get("type") == "message"]
    if len(messages) != 1 or any(item.get("type") not in ("message", "reasoning") for item in output):
        result["error_category"] = "output_message_count"; return result
    message = messages[0]
    content = message.get("content")
    if message.get("role") != "assistant" or message.get("status") != "completed" or not isinstance(content, list):
        result["error_category"] = "malformed_message"; return result
    if any(isinstance(part, dict) and part.get("type") == "refusal" for part in content):
        result["status"] = "refused"; return result
    if len(content) != 1 or not isinstance(content[0], dict) or content[0].get("type") != "output_text":
        result["error_category"] = "output_text_count"; return result
    def unique_object(pairs):
        obj = {}
        for key, value in pairs:
            if key in obj:
                raise ValueError("Duplicate JSON field")
            obj[key] = value
        return obj
    try:
        answer = json.loads(content[0].get("text"), object_pairs_hook=unique_object)
        if not isinstance(answer, dict) or set(answer) != {"label"} or type(answer["label"]) is not str or answer["label"] not in CLASSES:
            raise ValueError("Expected exactly one allowed label")
    except (ValueError, TypeError):
        result["error_category"] = "invalid_label_json"; return result
    result.update(status="answered", prediction=answer["label"])
    return result

def parsed_record(record):
    if record["state"] == "returned":
        result = parse_answer(record["raw_response"])
    else:
        result = {"status": "pending" if record["state"] in ("intent", "uncertain") else "failed",
                  "prediction": None, "probability_valid": False, "confidence_valid": False,
                  "error_category": record.get("error_category", record["state"])}
    result.update(attempt_id=record["attempt_id"], attempt_latency_seconds=record.get("latency_seconds"),
                  end_to_end_seconds=record.get("image_elapsed_seconds"), cache_key=record["cache_key"])
    return result

def bounded_delay(retry_index, retry_after=None):
    # Numeric Retry-After is bounded. HTTP dates are supported without logging headers.
    hint = 0.0
    try:
        hint = float(retry_after)
    except (TypeError, ValueError):
        if retry_after:
            from email.utils import parsedate_to_datetime
            try:
                hint = max(0, (parsedate_to_datetime(retry_after) - datetime.now(timezone.utc)).total_seconds())
            except (TypeError, ValueError, OverflowError):
                pass
    if not math.isfinite(hint):
        hint = 0
    return min(MAX_BACKOFF_SECONDS, max(hint, 2 ** retry_index + random.uniform(0, 1)))

def classify(png, client_factory=live_client, sleeper=time.sleep, on_record=None):
    key = cache_key(png)
    with runner_lock():
        previous = records_for(key)
        if previous:
            last = previous[-1]
            retry_uncertain = last["state"] in ("intent", "uncertain") and RETRY_UNCERTAIN_REQUESTS
            retry_rate_limit = last["state"] == "rate_limit" and RETRY_RATE_LIMITED
            if not (retry_uncertain or retry_rate_limit):
                return {**parsed_record(last), "cache_reused": True}
        if attempt_count() >= MAX_API_ATTEMPTS:
            return {"status": "pending", "prediction": None, "error_category": "budget_exhausted", "cache_reused": False}
        # Construct lazily only when a real attempt is permitted. Initialization failure reserves nothing.
        client = client_factory()
        started = time.perf_counter()
        try:
            for retry_index in range(MAX_ATTEMPTS_PER_IMAGE):
                existing_count = attempt_count()
                if existing_count >= MAX_API_ATTEMPTS:
                    break
                attempt_id = f"{existing_count + 1:08d}_{uuid.uuid4().hex}_{key}"
                path = OUTPUT_DIR / "attempts" / (attempt_id + ".json")
                record = {"attempt_id": attempt_id, "cache_key": key, "state": "intent", "created_at": utc_now(),
                          "request_semantics": request_semantics(), "image_sha256": hashlib.sha256(png).hexdigest(),
                          "rate_limit_resume_opt_in": RETRY_RATE_LIMITED,
                          "uncertain_retry_opt_in": RETRY_UNCERTAIN_REQUESTS}
                atomic_json(path, record)  # Reserve budget BEFORE calling the network.
                if on_record is not None:
                    on_record(record)
                begin = time.perf_counter()
                delay = None
                try:
                    response = client.responses.create(**request_body(png))
                    # Persist structured response before parsing. No request headers/client/key are stored.
                    record.update(state="returned", raw_response=response.model_dump(mode="json", warnings=False))
                except openai.RateLimitError as error:
                    record.update(state="rate_limit", error_category="rate_limit", http_status=429)
                    delay = bounded_delay(retry_index, error.response.headers.get("retry-after"))
                except (openai.APITimeoutError, openai.APIConnectionError):
                    record.update(state="uncertain", error_category="transport_or_timeout", http_status=None)
                    if RETRY_UNCERTAIN_REQUESTS:
                        delay = bounded_delay(retry_index)
                except openai.APIStatusError as error:
                    status = error.status_code
                    record.update(state="uncertain" if status >= 500 or status == 408 else "permanent_error",
                                  error_category="server_error" if status >= 500 else "http_error", http_status=status)
                    if record["state"] == "uncertain" and RETRY_UNCERTAIN_REQUESTS:
                        delay = bounded_delay(retry_index)
                except Exception:
                    # Unknown failures may have happened after dispatch. Never print raw exception text.
                    record.update(state="uncertain", error_category="unexpected_client_error", http_status=None)
                record.update(latency_seconds=time.perf_counter() - begin,
                              image_elapsed_seconds=time.perf_counter() - started, finished_at=utc_now())
                atomic_json(path, record)
                if on_record is not None:
                    on_record(record)
                if delay is None or retry_index + 1 >= MAX_ATTEMPTS_PER_IMAGE or attempt_count() >= MAX_API_ATTEMPTS:
                    return {**parsed_record(record), "cache_reused": False}
                record["retry_delay_requested_seconds"] = delay
                atomic_json(path, record)
                sleep_started = time.perf_counter()
                try:
                    sleeper(delay)
                finally:
                    record["retry_delay_seconds"] = time.perf_counter() - sleep_started
                    record["image_elapsed_seconds"] = time.perf_counter() - started
                    atomic_json(path, record)
            if records_for(key):
                return {**parsed_record(records_for(key)[-1]), "cache_reused": False}
            return {"status": "pending", "prediction": None, "error_category": "budget_exhausted", "cache_reused": False}
        finally:
            client.close()


def attempt_cost_usd(record):
    usage = (record.get("raw_response") or {}).get("usage") or {}
    details = usage.get("input_tokens_details") or {}
    cached = details.get("cached_tokens")
    written = details.get("cache_write_tokens")
    tokens_in, tokens_out = usage.get("input_tokens"), usage.get("output_tokens")
    if TOKEN_RATES_USD_PER_MILLION is None or not all(type(v) is int and v >= 0 for v in (tokens_in, tokens_out, cached, written)):
        return None
    if cached + written > tokens_in:
        return None
    rates = TOKEN_RATES_USD_PER_MILLION
    assert set(rates) == {"uncached_input", "cached_input", "cache_write", "output"}
    assert all(type(v) in (int, float) and math.isfinite(v) and v >= 0 for v in rates.values())
    # No cache options are sent, so writes default to 30 minutes. Respect returned
    # TTL detail when present; 12-hour writes cost twice the 30-minute rate.
    breakdown = details.get("cache_write_tokens_details")
    weighted_writes = written
    if breakdown is not None:
        short, long = breakdown.get("ttl_30m_tokens"), breakdown.get("ttl_12h_tokens")
        if not all(type(v) is int and v >= 0 for v in (short, long)) or short + long != written:
            return None
        weighted_writes = short + 2 * long
    # Input includes image tokens; output includes reasoning tokens. Neither is added twice.
    return ((tokens_in-cached-written)*rates["uncached_input"] + cached*rates["cached_input"] + weighted_writes*rates["cache_write"] + tokens_out*rates["output"])/1e6


PROJECTION_NOTE = "1000 times mean total attempt cost per unique image with fully priced history; includes observed retries/refusals/failures. Assumes same retry mix, not 1000 successful answers. Excludes images with any unknown usage; subset may be biased. No observations means unavailable."


def usage_summary(records):
    # Unique persisted attempts, including refusals/incomplete responses and all retries.
    records = list({record["attempt_id"]: record for record in records}.values())
    totals = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0,
              "cached_input_tokens": 0, "cache_write_tokens": 0, "reasoning_tokens": 0}
    coverage = {key: 0 for key in totals}
    latencies = []; costs = []; by_image = {}
    for record in records:
        if record.get("latency_seconds") is not None:
            latencies.append(record["latency_seconds"])
        usage = (record.get("raw_response") or {}).get("usage") or {}
        values = {**{key: usage.get(key) for key in ["input_tokens", "output_tokens", "total_tokens"]},
                  "cached_input_tokens": (usage.get("input_tokens_details") or {}).get("cached_tokens"),
                  "cache_write_tokens": (usage.get("input_tokens_details") or {}).get("cache_write_tokens"),
                  "reasoning_tokens": (usage.get("output_tokens_details") or {}).get("reasoning_tokens")}
        for key, value in values.items():
            if type(value) is int and value >= 0:
                totals[key] += value; coverage[key] += 1
        cost = attempt_cost_usd(record)
        costs.append(cost)
        by_image.setdefault(record["cache_key"], []).append(cost)
    observed = [sum(values) for values in by_image.values() if all(value is not None for value in values)]
    known = [cost for cost in costs if cost is not None]
    sessions_path = OUTPUT_DIR / "stage_sessions.json"
    sessions = read_json(sessions_path) if sessions_path.exists() else []
    return {"unique_attempts": len(records), "unique_attempted_images": len(by_image),
            "token_totals_available": totals, "usage_field_attempt_coverage": coverage,
            "mean_attempt_latency_seconds": float(np.mean(latencies)) if latencies else None,
            "sum_attempt_latency_seconds": float(sum(latencies)), "latency_available_attempts": len(latencies),
            "sum_recorded_stage_runtime_seconds": sum(s["runtime_seconds"] for s in sessions),
            "estimated_cost_usd": sum(known) if known else None, "priced_attempts": len(known),
            "unpriced_attempts": len(records)-len(known), "cost_complete": bool(records) and len(known) == len(records),
            "projected_1000_images_usd": 1000*sum(observed)/len(observed) if observed else None,
            "projection_observed_images": len(observed),
            "projection_note": PROJECTION_NOTE,
            "rates_usd_per_million": TOKEN_RATES_USD_PER_MILLION,
            "pricing_source": PRICING_SOURCE, "pricing_checked_date": PRICING_CHECKED_DATE,
            "cost_status": "standard rate estimate, not billing or a monetary cap; known subtotal when usage incomplete",
            "usage_note": "Image tokens included in input; reasoning included in output. Missing usage, including cache-write counts, is unknown, never free. Writes are removed from ordinary input before pricing; absent TTL detail uses the documented default 30-minute write rate, with returned 12-hour counts priced at twice that rate. Stage overlap/cache reads counted once."}


class CostAccounting:
    def __init__(self, records):
        self.attempt_costs = {}
        self.image_costs = {}
        self.known_cost = 0.0
        self.priced_attempts = 0
        self.observed_cost = 0.0
        self.observed_images = 0
        for record in records:
            self.update(record)

    def update(self, record):
        attempt_id, key = record["attempt_id"], record["cache_key"]
        cost = attempt_cost_usd(record)
        total, unknown = self.image_costs.get(key, (0.0, 0))
        if key in self.image_costs and not unknown:
            self.observed_cost -= total
            self.observed_images -= 1
        if attempt_id in self.attempt_costs:
            previous = self.attempt_costs[attempt_id]
            if previous is None:
                unknown -= 1
            else:
                total -= previous
                self.known_cost -= previous
                self.priced_attempts -= 1
        self.attempt_costs[attempt_id] = cost
        if cost is None:
            unknown += 1
        else:
            total += cost
            self.known_cost += cost
            self.priced_attempts += 1
        self.image_costs[key] = (total, unknown)
        if not unknown:
            self.observed_cost += total
            self.observed_images += 1

    def summary(self):
        return {"unique_attempts": len(self.attempt_costs), "priced_attempts": self.priced_attempts,
                "estimated_cost_usd": self.known_cost if self.priced_attempts else None,
                "projected_1000_images_usd": 1000*self.observed_cost/self.observed_images if self.observed_images else None,
                "projection_observed_images": self.observed_images}


def cost_progress(accounting=None):
    summary = accounting.summary() if accounting is not None else usage_summary(attempt_records())
    def usd(value):
        return "unavailable" if value is None else f"${value:.6f}"
    return {"known estimated USD": usd(summary["estimated_cost_usd"]),
            "projected 1000 USD": usd(summary["projected_1000_images_usd"]),
            "projection observed images": summary["projection_observed_images"],
            "priced attempts": f"{summary['priced_attempts']}/{summary['unique_attempts']}"}


def run_stage(manifest, enabled):
    if not enabled:
        print(manifest["stage"], "disabled; analysis will use any saved results only.")
        return
    if manifest["split"] == "test":
        check_freeze()
    elif freeze_path().exists():
        raise RuntimeError("Development calls are closed after protocol freeze. Use cached analysis or a new study directory.")
    if manifest["protocol"] != protocol():
        raise RuntimeError("Manifest does not match current protocol.")
    started = time.perf_counter()
    counts = {}
    try:
        accounting = CostAccounting(attempt_records())
        progress = tqdm(manifest["rows"], desc=manifest["stage"])
        for row in progress:
            png = (OUTPUT_DIR / "images" / (row["image_sha256"] + ".png")).read_bytes()
            assert cache_key(png) == row["cache_key"]
            result = classify(png, on_record=accounting.update)
            counts[result["status"]] = counts.get(result["status"], 0) + 1
            progress.set_postfix(cost_progress(accounting))
            # Stop before traversing hundreds of unattempted images after the cap.
            if result.get("error_category") == "budget_exhausted":
                break
    finally:
        log_path = OUTPUT_DIR / "stage_sessions.json"
        sessions = read_json(log_path) if log_path.exists() else []
        sessions.append({"stage": manifest["stage"], "ended_at": utc_now(),
                         "runtime_seconds": time.perf_counter() - started, "counts": counts})
        atomic_json(log_path, sessions)
    print("Stage session:", counts, "Lifetime attempts:", attempt_count(), "/", MAX_API_ATTEMPTS)
    print("Cumulative cost/projection:", cost_progress(accounting))
    print(PROJECTION_NOTE)

## 7. Explicit development smoke and pilot

Enable one development flag in configuration, rerun configuration, then run through this cell. Inspect saved summaries and galleries before finalizing. The default 20-attempt cap deliberately cannot complete pilot100 or evaluation1000. A refused or failed cached item is not retried on rerun. An unanswered image due solely to the cap remains eligible when you explicitly raise the cap. HTTP 429 exhaustion remains cached on ordinary reruns. Deliberately set `RETRY_RATE_LIMITED=True` to resume it under the same persistent cap, then turn the switch off again.

In [ ]:
run_stage(smoke_manifest, RUN_SMOKE10)
run_stage(pilot_manifest, RUN_PILOT100)

## 8. Freeze the prompt and procedure before opening test data

Set both development flags off and `FREEZE_PROTOCOL=True` after development review. Execute this cell to save the exact protocol. It is idempotent for the same protocol and refuses changes. No test loading occurs here. Thereafter development calls are closed for this study. For a zero-development run you may deliberately freeze immediately, but document that limitation when interpreting results.

In [ ]:
if FREEZE_PROTOCOL:
    if freeze_path().exists():
        check_freeze()
        print("Existing identical protocol freeze retained.")
    else:
        atomic_json(freeze_path(), {"frozen_at": utc_now(), "protocol": protocol()})
        print("Prompt and procedure frozen. Test may now be enabled in a separate explicit step.")
else:
    print("Protocol freeze is", "already saved" if freeze_path().exists() else "not yet requested", ".")

## 9. Explicit held-out test evaluation

Only after a saved matching freeze, enable `RUN_EVALUATION1000=True` and deliberately raise the lifetime cap if desired. This stage samples 100 original test images per class with a fixed seed. `RUN_FULL10000=True` is a separate optional authorization for all 1,000 per class. Default execution does neither. Unchanged protocol and sample identities are required for a resume. Never revise the question based on test errors and continue calling the same study an untouched evaluation.

In [ ]:
if RUN_EVALUATION1000 or RUN_FULL10000:
    check_freeze()  # Must pass BEFORE original test data are loaded.
    test_data = CIFAR10(root=str(DATA_ROOT), train=False, download=True)
    assert test_data.classes == CLASSES
    if RUN_EVALUATION1000:
        evaluation_manifest = prepare_stage("evaluation1000", test_data, "test")
        run_stage(evaluation_manifest, True)
    if RUN_FULL10000:
        full_manifest = prepare_stage("full10000", test_data, "test")
        run_stage(full_manifest, True)
else:
    print("Test evaluation disabled. Original test data were not loaded by this cell.")

## 10. Failure-aware metrics, uncertainty and visualization

Definitions used below:

- **Requested:** planned stage size. **Attempted:** distinct images with any persisted request intent. **Answered:** valid allowed returned choices. **Failed:** permanent, rate-limit-exhausted or malformed-choice outcomes. **Refused:** explicit model refusal. **Pending:** unattempted or uncertain images. Coverage is answered/requested; attempted pending requests remain in the attempted denominator.
- **Attempted-image accuracy:** correct/attempted, treating refused, failed and uncertain attempts as not correct. **Answered-only accuracy:** correct/answered, conditional on obtaining a classification. **Planned-set lower bound:** correct/requested while incomplete, not completed accuracy. A completed planned result is reported only when no pending rows remain.
- Per-class precision, recall and F1 use every planned truth label; absent predictions reduce recall. Zero division is defined as zero, including when a class has no answers. Macro means average ten classes; weighted means weight by planned support. `conditional_recall` and answered counts make coverage visible. A 10x10 confusion matrix contains **valid answers only**, normalized within answered truth classes; separate class failure/refusal/pending columns carry missing cases. Rows with no answers plot as zero and must not be read as observed error rates.
- **Confidence and class probabilities unavailable:** label-only Responses does not supply either. Confidence histograms, reliability plots, ECE, multiclass Brier and NLL are skipped with an explanation; their interfaces remain null. Error galleries retain original images and display confidence as unavailable.
- Approximate 95% Wilson intervals use an independent Bernoulli model; the balanced stratified sample and incomplete/nonrandom response process limit their interpretation. In particular, 10/10 or 100/100 cannot establish 99% full-test accuracy.

Runtime includes measured stage execution and per-attempt latency; image elapsed time includes retry delay within that classification invocation. Total stage runtime can include cache reads and is accumulated across sessions. Unobserved crash time is unavailable. Usage and standard-rate cost estimates are aggregated over unique global attempts, never summed across overlapping stages. Cached-input/reasoning-token details remain in each raw response and exports. No returned usage is assumed for failed/uncertain attempts. Image tokens are already in input usage; reasoning tokens are already in output usage. Neither is double counted. Known estimated cost and the 1000-image projection disclose usage coverage, the fully priced unique-image denominator and observed-retry assumption; unknown usage remains unknown. Model aliases and provider processing can change over time; the fixed request does not guarantee an immutable model snapshot or deterministic answers.

In [ ]:
def stage_frame(manifest):
    # One row per planned image, including failed/refused/unattempted records.
    all_records = attempt_records()
    by_key = {}
    for record in all_records:
        by_key.setdefault(record["cache_key"], []).append(record)
    rows = []
    for row in manifest["rows"]:
        history = by_key.get(row["cache_key"], [])
        result = parsed_record(history[-1]) if history else {"status": "pending", "prediction": None,
                                                            "probability_valid": False, "confidence_valid": False}
        rows.append({**row, **result, "attempted": bool(history), "attempt_count": len(history),
                     "latency_attempts_available_count": sum(r.get("latency_seconds") is not None for r in history),
                     "image_attempt_latency_total_seconds": sum(r.get("latency_seconds", 0) for r in history),
                     "image_retry_delay_total_seconds": sum(r.get("retry_delay_seconds", 0) for r in history)})
    frame = pd.DataFrame(rows)
    frame["correct"] = (frame["status"] == "answered") & (frame["prediction"] == frame["truth"])
    return frame


def wilson(successes, total):
    if not total:
        return None
    z = 1.959963984540054
    p = successes / total
    center = (p + z*z/(2*total)) / (1 + z*z/total)
    half = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / (1 + z*z/total)
    return [max(0, center-half), min(1, center+half)]


def stage_metrics(frame):
    n = len(frame)
    answered = frame["status"] == "answered"
    attempted = frame["attempted"]
    a = int(answered.sum()); t = int(attempted.sum()); correct = int(frame["correct"].sum())
    metrics = {"requested": n, "attempted": t, "answered": a,
               "failed": int((frame.status == "failed").sum()), "refused": int((frame.status == "refused").sum()),
               "pending": int((frame.status == "pending").sum()), "correct": correct,
               "coverage": a/n if n else None, "attempted_image_accuracy": correct/t if t else None,
               "answered_only_accuracy": correct/a if a else None,
               "planned_set_accuracy_lower_bound": correct/n if n else None,
               "completed_planned_accuracy": correct/n if n and not (frame.status == "pending").any() else None,
               "attempted_accuracy_wilson95": wilson(correct, t), "answered_accuracy_wilson95": wilson(correct, a)}
    observed_elapsed = frame.loc[attempted, "image_attempt_latency_total_seconds"] + frame.loc[attempted, "image_retry_delay_total_seconds"]
    timed = frame.loc[attempted, "latency_attempts_available_count"] > 0
    metrics["image_work_timing_covered_count"] = int(timed.sum())
    metrics["mean_observed_image_work_seconds"] = float(observed_elapsed[timed].mean()) if timed.any() else None
    metrics["total_observed_image_work_seconds"] = float(observed_elapsed.sum())
    metrics["latency_note"] = "Known attempt latency plus measured retry sleep across image resumes. Mean includes only images with timing; totals are lower bounds when interrupted work lacks timing. Between-session idle time unavailable."
    per_class = []
    for label in CLASSES:
        truth = frame.truth == label
        predicted = answered & (frame.prediction == label)
        tp = int((truth & predicted).sum()); support = int(truth.sum()); predicted_n = int(predicted.sum())
        precision = tp/predicted_n if predicted_n else 0.0
        recall = tp/support if support else 0.0
        conditional_support = int((truth & answered).sum())
        per_class.append({"class": label, "precision": precision, "recall": recall,
                          "f1": 2*precision*recall/(precision+recall) if precision+recall else 0.0,
                          "support": support, "answered": conditional_support,
                          "conditional_recall": tp/conditional_support if conditional_support else None,
                          "failed": int((truth & (frame.status == "failed")).sum()),
                          "refused": int((truth & (frame.status == "refused")).sum()),
                          "pending": int((truth & (frame.status == "pending")).sum())})
    classes = pd.DataFrame(per_class).set_index("class")
    for metric in ("precision", "recall", "f1"):
        metrics["macro_" + metric] = float(classes[metric].mean())
        metrics["weighted_" + metric] = float(np.average(classes[metric], weights=classes.support)) if n else None
    # 10x10 matrices are conditional on valid answers; failures are separately counted above.
    raw = pd.crosstab(frame.loc[answered, "truth"], frame.loc[answered, "prediction"]).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
    normalized = raw.div(raw.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
    valid = frame.loc[answered & frame.probability_valid].copy()
    metrics.update(probability_valid_count=len(valid), probability_invalid_answered=a-len(valid),
                   api_confidence_valid_count=int((answered & frame.confidence_valid).sum()))
    bins = []
    if len(valid):
        selected = valid.selected_probability.to_numpy(dtype=float)
        outcomes = valid.correct.to_numpy(dtype=float)
        vectors = np.array([[p[label] for label in CLASSES] for p in valid.probabilities])
        truth_index = np.array([CLASSES.index(label) for label in valid.truth])
        targets = np.eye(10)[truth_index]
        metrics["multiclass_brier"] = float(np.mean(np.sum((vectors-targets)**2, axis=1)))
        # Zero assigned to the true class gives infinite NLL, encoded as null plus zero-count.
        true_probs = vectors[np.arange(len(valid)), truth_index]
        zeros = int((true_probs == 0).sum())
        metrics["nll_zero_true_probability_count"] = zeros
        metrics["multiclass_nll"] = float(-np.log(true_probs).mean()) if not zeros else None
        metrics["nll_infinite"] = bool(zeros)
        metrics["choice_argmax_disagreements"] = int(valid.choice_argmax_disagreement.sum())
        ece = 0.0
        for index in range(CALIBRATION_BINS):
            low, high = index/CALIBRATION_BINS, (index+1)/CALIBRATION_BINS
            mask = (selected >= low) & ((selected < high) if index+1 < CALIBRATION_BINS else (selected <= high))
            count = int(mask.sum())
            mean = float(selected[mask].mean()) if count else None
            accuracy = float(outcomes[mask].mean()) if count else None
            if count:
                ece += count/len(valid) * abs(mean-accuracy)
            bins.append({"lower_inclusive": low, "upper": high, "upper_inclusive": index+1 == CALIBRATION_BINS,
                         "count": count, "mean_selected_probability": mean, "accuracy": accuracy})
        metrics["ece_selected_probability"] = ece
    else:
        metrics.update(multiclass_brier=None, multiclass_nll=None, ece_selected_probability=None,
                       choice_argmax_disagreements=None)
    metrics["calibration_bins"] = CALIBRATION_BINS
    return metrics, classes, raw, normalized, pd.DataFrame(bins)


def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=170, bbox_inches="tight")
    plt.show(); plt.close(fig)


def stage_plots(frame, metrics, classes, raw, normalized, calibration, folder, title):
    answered = frame[frame.status == "answered"]
    if answered.empty:
        print(title, ": no valid classifications; probability/confusion/gallery plots skipped.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(raw, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False)
    sns.heatmap(normalized, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=axes[1])
    for axis, subtitle in zip(axes, ["Raw counts", "Row fractions among answered images"]):
        axis.set_title(subtitle); axis.set_xlabel("Returned choice"); axis.set_ylabel("True class")
        axis.tick_params(axis="x", rotation=45); axis.tick_params(axis="y", rotation=0)
    fig.suptitle(f"{title}: conditional confusion, {len(answered)}/{len(frame)} answered")
    save_figure(fig, folder / "confusion.png")
    fig, axis = plt.subplots(figsize=(11, 4.5))
    classes[["precision", "recall", "f1"]].plot.bar(ax=axis, ylim=(0, 1), rot=35)
    axis.set_title(f"{title}: planned-set metrics (missing choices reduce recall)")
    axis.set_ylabel("Score"); save_figure(fig, folder / "class_metrics.png")
    errors = answered[~answered.correct].groupby(["truth", "prediction"]).size().sort_values(ascending=False).head(12)
    if len(errors):
        fig, axis = plt.subplots(figsize=(10, 4))
        errors.rename(index=lambda x: x).plot.bar(ax=axis, color="#bf6253")
        axis.set_title(f"{title}: most frequent error pairs (truth, prediction)"); axis.set_ylabel("Images")
        axis.tick_params(axis="x", rotation=35); save_figure(fig, folder / "top_errors.png")
    print(title, ": confidence histograms, reliability/calibration, Brier and NLL unavailable: label-only Responses supplies no confidence or class probabilities.")
    valid_confidence = answered[answered.confidence_valid]
    if len(valid_confidence):
        fig, axis = plt.subplots(figsize=(8, 4))
        for correct, color in [(True, "#287c70"), (False, "#bf6253")]:
            values = valid_confidence.loc[valid_confidence.correct == correct, "api_confidence"].astype(float)
            if len(values):
                axis.hist(values, bins=np.linspace(0, 1, 11), alpha=.65, label=f"{'Correct' if correct else 'Incorrect'} (n={len(values)})", color=color)
        axis.set(xlabel="API confidence (separate from selected probability)", ylabel="Images", title=title)
        axis.legend(); save_figure(fig, folder / "api_confidence.png")
    if len(calibration):
        observed = calibration[calibration["count"] > 0]
        fig, axis = plt.subplots(figsize=(6, 5))
        axis.plot([0, 1], [0, 1], "--", color="gray")
        axis.scatter(observed.mean_selected_probability, observed.accuracy, s=30+observed["count"]*4)
        axis.set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean probability of returned choice", ylabel="Fraction correct",
                 title=f"{title}: reliability (n={metrics['probability_valid_count']}, {CALIBRATION_BINS} bins)")
        save_figure(fig, folder / "calibration.png")
    focus = classes.loc[["cat", "bird", "dog"]]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    focus[["precision", "recall", "f1"]].plot.bar(ax=axes[0], ylim=(0, 1), rot=0)
    axes[0].set_title("Cat, bird and dog: planned-set metrics")
    sns.heatmap(raw.loc[["cat", "dog"], ["cat", "dog"]], annot=True, fmt="d", cmap="Oranges", ax=axes[1], cbar=False)
    axes[1].set_title("Cat/dog choices only (other choices excluded)")
    axes[1].set_xlabel("Returned choice"); axes[1].set_ylabel("True class")
    fig.suptitle(title); save_figure(fig, folder / "cat_bird_dog.png")
    for correct, name in [(True, "correct"), (False, "incorrect")]:
        subset = answered[answered.correct == correct].head(10)
        if subset.empty:
            continue
        columns = min(5, len(subset))
        rows = math.ceil(len(subset)/columns)
        fig, axes = plt.subplots(rows, columns, figsize=(2.4*columns, 2.7*rows), squeeze=False)
        for axis in axes.flat:
            axis.axis("off")
        for axis, (_, row) in zip(axes.flat, subset.iterrows()):
            image = Image.open(OUTPUT_DIR / "images" / (row.image_sha256 + ".png"))
            axis.imshow(image, interpolation="nearest")
            confidence = f"{row.api_confidence:.3f}" if row.confidence_valid else "unavailable"
            axis.set_title(f"Truth: {row.truth}\nChoice: {row.prediction}\nAPI confidence: {confidence}", fontsize=9)
        fig.suptitle(f"{title}: {name}; original pixels enlarged only for display")
        save_figure(fig, folder / (name + "_gallery.png"))


def educational_summary(stage, split, metrics, classes, frame):
    lines = [f"### {stage} ({split})", "Confidence and class probabilities are unavailable for label-only Responses; confidence/calibration plots, ECE, Brier and NLL are skipped.", f"Planned {metrics['requested']}; attempted {metrics['attempted']}; answered {metrics['answered']}; "
             f"failed {metrics['failed']}; refused {metrics['refused']}; pending {metrics['pending']}. Coverage: {metrics['coverage']:.1%}."]
    if not metrics["attempted"]:
        lines.append("No measured Responses results yet. Enable a development stage deliberately to collect evidence. No accuracy or superiority claim can be made.")
        return "\n\n".join(lines)
    def pct(value):
        return "unavailable" if value is None else f"{value:.2%}"
    lines.append(f"Failure-aware attempted-image accuracy: {pct(metrics['attempted_image_accuracy'])}; answered-only conditional accuracy: "
                 f"{pct(metrics['answered_only_accuracy'])}. Planned-set lower bound: {pct(metrics['planned_set_accuracy_lower_bound'])}.")
    if metrics["pending"]:
        lines.append("This run is incomplete. The lower bound is not completed evaluation accuracy; class rankings below are provisional and depend on coverage.")
    if metrics["answered"]:
        ranked = classes.sort_values("f1", kind="stable")
        easiest = ", ".join(ranked.index[ranked.f1 == ranked.f1.max()])
        hardest = ", ".join(ranked.index[ranked.f1 == ranked.f1.min()])
        lines.append(f"Measured planned-set F1: easiest {easiest}; hardest {hardest}. Missing predictions count against recall.")
        lines.append(f"Cat F1 {classes.loc['cat', 'f1']:.3f}; bird F1 {classes.loc['bird', 'f1']:.3f}. "
                     "Historical supervised CNN cat F1 0.626 and bird F1 0.688 are descriptive references.")
        pairs = frame[(frame.status == "answered") & ~frame.correct].groupby(["truth", "prediction"]).size().sort_values(ascending=False)
        if len(pairs):
            lines.append("Most common measured errors: " + "; ".join(f"{a} to {b}: {int(n)}" for (a, b), n in pairs.head(3).items()) + ".")
        cat_dog = int((((frame.truth == "cat") & (frame.prediction == "dog")) | ((frame.truth == "dog") & (frame.prediction == "cat"))).sum())
        lines.append(f"Cat/dog swaps: {cat_dog}. Visual similarity is a hypothesis to inspect in the error gallery, not an established cause.")
    observed_confidence = frame[(frame.status == "answered") & frame.confidence_valid]
    confidence_findings = []
    for correctness, label in [(True, "correct"), (False, "incorrect")]:
        values = observed_confidence.loc[observed_confidence.correct == correctness, "api_confidence"] if len(observed_confidence) else []
        if len(values):
            confidence_findings.append(f"{label} mean {float(np.mean(values)):.3f} (n={len(values)})")
    if confidence_findings:
        lines.append("Measured API confidence by correctness: " + "; ".join(confidence_findings) + ". This is a descriptive confidence comparison, not a calibrated probability guarantee.")
    interval = metrics["attempted_accuracy_wilson95"]
    if interval:
        lines.append(f"Approximate 95% Wilson interval for attempted-image correctness: {interval[0]:.2%} to {interval[1]:.2%}. "
                     "This is descriptive under an independent Bernoulli approximation; balanced class sampling, finite population and nonresponse limit its interpretation.")
    if split == "test":
        gain = metrics["attempted_image_accuracy"] - HISTORICAL_CNN_ACCURACY
        lines.append(f"Observed attempted-subset difference from historical 79.08% CNN accuracy: {100*gain:+.2f} percentage points. "
                     "This compares a current subset to a historical full 10,000-image supervised CNN result; it is neither paired nor causal.")
    else:
        lines.append("These training validation images are development evidence only; do not use them as the test comparison.")
    if metrics["probability_valid_count"]:
        lines.append(f"Probability-valid answers: {metrics['probability_valid_count']}/{metrics['answered']}; selected-choice ECE "
                     f"{metrics['ece_selected_probability']:.4f}; Brier {metrics['multiclass_brier']:.4f}. "
                     f"Choice/argmax disagreements: {metrics['choice_argmax_disagreements']}. API confidence is analyzed separately and is not correctness.")
    else:
        lines.append("No complete valid probability vectors; calibration, Brier and NLL are unavailable.")
    lines.append("99% remains an aspiration. Tiny 32x32 images can lose distinguishing detail; zero-shot pretrained knowledge can help but may include CIFAR contamination. "
                 "A small or incomplete sample cannot establish 99% full-test performance. More evidence requires a deliberate budget increase and untouched test evaluation under the frozen protocol.")
    return "\n\n".join(lines)

## 11. Analyze cached measurements and export

This cell makes **no API requests**, even when paid flags are on. It loads every saved stage separately under the matching protocol, computes metrics, writes CSV/JSON, plots only when observations exist, and generates the narrative from actual records. Per-stage files include predictions, class metrics, raw/normalized confusion, calibration bins, figures and Markdown findings. Global files include unique attempts with usage/latency, usage/runtime, metrics and the final summary.

The supervised CNN learned CIFAR training labels; Responses is zero-shot with respect to this notebook's labeled examples but may have pretrained knowledge, including unknown exposure to CIFAR images. Historical full-test CNN results versus a current Responses subset are descriptive, not paired or causal evidence. Visual similarity (especially cat/dog), background context, label conventions and 32x32 information loss are plausible error explanations to check in the gallery. Label-only Responses supplies no confidence or class probabilities, so confidence/calibration comparisons are unavailable. These are hypotheses until measurements support them; 99% is not guaranteed by larger models or stronger pretraining.

In [ ]:
all_summaries = []
all_metrics = {}
for manifest_path in sorted((OUTPUT_DIR / "manifests").glob("*.json")):
    manifest = read_json(manifest_path)
    if manifest["protocol"] != protocol():
        raise RuntimeError("Analysis stopped: saved manifest differs from current protocol. Restore it before analyzing.")
    if manifest["split"] == "test":
        check_freeze()
    name = manifest["stage"]
    frame = stage_frame(manifest)
    metrics, classes, raw, normalized, calibration = stage_metrics(frame)
    folder = OUTPUT_DIR / "exports" / name
    folder.mkdir(exist_ok=True)
    exported = frame.copy()
    for column in ["probabilities", "probabilities_raw"]:
        if column in exported:
            exported[column] = exported[column].map(lambda value: json.dumps(value) if isinstance(value, (dict, list)) else None)
    exported.to_csv(folder / "predictions.csv", index=False)
    classes.to_csv(folder / "per_class_metrics.csv")
    raw.to_csv(folder / "confusion_raw.csv")
    normalized.to_csv(folder / "confusion_normalized.csv")
    calibration.to_csv(folder / "calibration_bins.csv", index=False)
    atomic_json(folder / "metrics.json", metrics)
    summary = educational_summary(name, manifest["split"], metrics, classes, frame)
    (folder / "summary.md").write_text(summary, encoding="utf-8")
    display(Markdown(summary)); display(classes)
    stage_plots(frame, metrics, classes, raw, normalized, calibration, folder, name)
    all_metrics[name] = metrics
    all_summaries.append(summary)

unique_records = attempt_records()
global_usage = usage_summary(unique_records)
atomic_json(OUTPUT_DIR / "exports" / "usage_and_runtime.json", global_usage)
# Safe structured raw attempt records retain answer/usage and sanitized errors.
# Unique attempts are exported once globally, even when multiple stages reuse them.
atomic_json(OUTPUT_DIR / "exports" / "attempts.json", unique_records)
usage_rows = []
for record in unique_records:
    usage = (record.get("raw_response") or {}).get("usage") or {}
    usage_rows.append({"attempt_id": record["attempt_id"], "cache_key": record["cache_key"], "state": record["state"],
                       "latency_seconds": record.get("latency_seconds"), "image_elapsed_seconds": record.get("image_elapsed_seconds"),
                       "estimated_cost_usd": attempt_cost_usd(record),
                       "usage_json": json.dumps(usage), "http_status": record.get("http_status"), "error_category": record.get("error_category")})
pd.DataFrame(usage_rows, columns=["attempt_id", "cache_key", "state", "latency_seconds", "image_elapsed_seconds", "estimated_cost_usd", "usage_json", "http_status", "error_category"]).to_csv(OUTPUT_DIR / "exports" / "attempts.csv", index=False)
final_text = "# Automatically generated measured-results summary\n\n" + "\n\n".join(all_summaries)
final_text += "\n\nUnique attempt usage/runtime (not summed across overlapping stages):\n\n```json\n" + json.dumps(global_usage, indent=2) + "\n```\n"
(OUTPUT_DIR / "exports" / "final_summary.md").write_text(final_text, encoding="utf-8")
atomic_json(OUTPUT_DIR / "exports" / "metrics.json", all_metrics)
print("Exports saved:", OUTPUT_DIR / "exports")
print("Cumulative cost/projection:", cost_progress())
print(global_usage["projection_note"])
print("No synthetic model results are included. An empty experiment has no measured API accuracy.")

## 12. Save, restore and share results safely

With Drive enabled, the entire results folder is persistent. Otherwise set `DOWNLOAD_RESULTS=True` below to download a ZIP **before the runtime expires**. The archive includes the request ledger and manifests, not a key. Avoid sharing any manually added files containing secrets. To resume in a new VM, upload/extract the archive to the configured `OUTPUT_DIR`, including `attempts`, `images`, `manifests`, `stage_sessions.json`, and the freeze file, then run with the same protocol. Do not import only CSV predictions: the durable request ledger is what prevents repeated calls. Do not reset the budget by changing directories mid-study; a new directory is a separate study with a separate cap and no deduplication guarantee.

For local analysis use all paid flags off. To enable live calls return to Colab and configure Secrets there. Re-running analysis exports never pays again for completed results. Uncertain requests require explicit duplicate-billing opt-in, and no tool here can prove whether an interrupted request was billed.

**Validation scope:** this notebook was structurally validated and executed offline with the existing local official dataset, install cell skipped, safe-default flags, network rejection and a Secrets stub that must not be called. Official SDK serialization/routing and synthetic label-only JSON/refusals/incomplete outputs/retries were tested with a mock transport only. Actual Colab package installation, Drive mounting, account availability, provider behavior, paid inference and real accuracy remain unmeasured. Synthetic test fixtures are stored outside notebook outputs and are never experiment results.

In [ ]:
# Optional local archive/download, disabled by default. Contains no API key.
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    archive = shutil.make_archive(str(OUTPUT_DIR.parent / (OUTPUT_DIR.name + "_backup")), "zip", OUTPUT_DIR)
    try:
        from google.colab import files
    except ImportError:
        print("Results archive:", archive)
    else:
        files.download(archive)